# Convergence of the conjugate gradient

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge: convergence of the conjugate gradient
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.sparse import diags
import random

In [ ]:
def eig_val_1d(nx, dx, i):
    a = (4/(dx**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2

def eig_vec(nx, dx, i):
    j = np.arange(1,nx+1)
    return np.sqrt(2/(nx+1)) * np.sin(j*i*np.pi/(nx+1))

def conjugate_gradient(a, b, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_norm_rk = []

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        ####print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")
        hist_norm_rk.append(norm_rk/norm_b)
        if norm_rk/norm_b < eps: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")

    return xk, hist_norm_rk

## Poisson equation

We want to solve the elliptic problem given by the Poisson equation in the 1D case subject to homogeneous Dirichlet boundary conditions:

$$
\left\{
\begin{aligned}
- & u''(x)  =  f(x) \quad \text{in} \; \Omega = [0,1] \quad \text{with} \; f(x)=1\\
  & u(0)  =  0 \; \text{and} \; u(1) = 0
\end{aligned}
\right.
$$

The finite difference discretisation makes it possible to reduce the problem to solving a linear system $Ax=b$.

We solve this system with the conjugate gradient algorithm, choosing the initial iterate $x_0 = 0$, for different right-hand sides.

In [ ]:
nx = 100
dx = 1/(nx+1)

# building the sparse matrix
diag = np.repeat(2/dx**2, nx)
diag_x = np.repeat(-1/dx**2, nx-1)
a  = diags([diag, diag_x, diag_x], [0, -1, 1])
print(f"1d case: nx = {nx}")
print(f"Matrix size: ({nx} x {nx})")

**Case of a right-hand side equal to an eigenvector of A**

In [ ]:
# random choice of eigen vector
eig_nb = np.ndarray.item(np.random.randint(nx, size=1))
b = eig_vec(nx, dx, eig_nb)

print(f"  Eigenvector number: {eig_nb}")

x, _ = conjugate_gradient(a, b)

How can we explain that the algorithm converges in 1 iteration?

**Case of a right-hand side equal to a linear combination of n eigenvectors of A**

In [ ]:
n = 10

b = np.zeros(nx)

for i in random.sample(range(1, nx+1), n):
    b = b + (i * eig_vec(nx, dx, i))

x, h = conjugate_gradient(a, b)

How can we explain that the algorithm converges in n iterations?

**Case of a right-hand side all of whose components are equal to 1**

In [ ]:
b = np.ones(nx)
u, h = conjugate_gradient(a, b, eps=1e-8)

How can we explain that the algorithm converges in $n_x/2$ iterations?

**Case of a right-hand side all of whose components are equal to 1 except one**

In [ ]:
b = np.ones(nx)
b[0] = 2 
u, h = conjugate_gradient(a, b)

How can we explain that the algorithm converges in nx iterations?